# 02. Исследовательский анализ данных

Исследуем время, цены, каналы, активность пользователей, популярность товаров и основные fashion-категории базовыми средствами pandas, NumPy и matplotlib.

## Подключение среды

В Colab проект читается с Google Drive. При локальном запуске используется текущая папка проекта. Все дальнейшие пути строятся от `PROJECT_ROOT`.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/fashion-recommender-system')
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Не найдена папка src: ' + str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Корень проекта:', PROJECT_ROOT)

В новой Colab-сессии зависимости устанавливаются из одного файла проекта. Локально этот шаг пропускается, если окружение уже подготовлено.

In [ ]:
import subprocess

if 'google.colab' in sys.modules:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')],
        check=True,
    )

### Библиотеки EDA

**Что делаем:** импортируем pandas, NumPy и matplotlib.  
**Зачем:** анализ и графики остаются прямо в notebook.  
**Что получим:** инструменты для следующих ячеек.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from fashion_recommender.data import load_articles, load_customers, load_transactions

### Пути к данным

**Что делаем:** задаём пути к CSV.  
**Зачем:** отделяем настройку путей от загрузки.  
**Что получим:** три входных пути.

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
TRANSACTIONS_PATH = RAW_DATA_DIR / 'transactions_train.csv'
ARTICLES_PATH = RAW_DATA_DIR / 'articles.csv'
CUSTOMERS_PATH = RAW_DATA_DIR / 'customers.csv'

### Загрузка таблиц

**Что делаем:** читаем transactions, articles и customers.  
**Зачем:** EDA использует проверенные типы и ключи.  
**Что получим:** три DataFrame.

In [ ]:
transactions = load_transactions(TRANSACTIONS_PATH)
articles = load_articles(ARTICLES_PATH)
customers = load_customers(CUSTOMERS_PATH)

## Покупки по времени

Месячная агрегация показывает общий тренд, а исходный DataFrame остаётся неизменным.

In [ ]:
purchases_by_month = transactions.groupby(transactions['t_dat'].dt.to_period('M')).size()
purchases_by_month.plot(figsize=(12, 4), title='Количество покупок по месяцам')
plt.xlabel('Месяц'); plt.ylabel('Покупки'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## Цена и канал продаж

Для цены показываем описательную статистику и центральные 99% значений, чтобы редкие выбросы не растягивали график.

In [ ]:
display(transactions['price'].describe())
price_limit = transactions['price'].quantile(0.99)
transactions.loc[transactions['price'] <= price_limit, 'price'].hist(bins=40, figsize=(10, 4))
plt.title('Распределение цены до 99-го процентиля'); plt.xlabel('Цена'); plt.show()
display(transactions['sales_channel_id'].value_counts(normalize=True).rename('share'))

## Активность пользователей и товаров

Считаем покупки на пользователя и уникальных покупателей на товар: это разные стороны неравномерности implicit feedback.

In [ ]:
purchases_per_user = transactions.groupby('customer_id').size()
customers_per_item = transactions.groupby('article_id')['customer_id'].nunique()
display(purchases_per_user.describe().rename('purchases_per_user'))
display(customers_per_item.describe().rename('customers_per_item'))
print('Пользователей с одной покупкой:', int((purchases_per_user == 1).sum()))
print('Товаров с одним покупателем:', int((customers_per_item == 1).sum()))

### Гистограмма активности

**Что делаем:** ограничиваем хвост только для графика.  
**Зачем:** несколько очень активных клиентов не должны скрыть основную массу.  
**Что получим:** читаемое распределение покупок.

In [ ]:
purchases_per_user.clip(upper=purchases_per_user.quantile(0.99)).hist(bins=40, figsize=(10, 4))
plt.title('Покупки на пользователя, значения ограничены 99-м процентилем'); plt.show()

## Популярные товары и категории

Для соединения выбираем только `article_id` и шесть нужных категорий. Полная широкая копия transactions + articles не создаётся.

In [ ]:
top_items = (
    transactions.groupby("article_id").size().rename("purchase_count").reset_index()
    .sort_values(
        ["purchase_count", "article_id"],
        ascending=[False, True],
        kind="mergesort",
    ).head(20).set_index("article_id")["purchase_count"]
)
top_items.plot(kind="bar", figsize=(12, 4), title="20 самых популярных товаров")
plt.ylabel('Покупки'); plt.tight_layout(); plt.show()
category_columns = ['product_group_name', 'product_type_name', 'colour_group_name', 'section_name', 'garment_group_name']
transaction_categories = transactions[['article_id']].merge(
    articles[['article_id', *category_columns]], on='article_id', how='left'
)

### Категории товаров

**Что делаем:** строим небольшие bar charts по категориям.  
**Зачем:** сравниваем популярность товарных признаков.  
**Что получим:** шесть компактных графиков.

In [ ]:
figure, axes = plt.subplots(3, 2, figsize=(15, 14))
for axis, column in zip(axes.ravel(), category_columns):
    category_counts = (
        transaction_categories.groupby(column).size().rename("purchase_count").reset_index()
        .sort_values(["purchase_count", column], ascending=[False, True], kind="mergesort")
        .head(12).set_index(column)["purchase_count"]
    )
    category_counts.plot(kind="bar", ax=axis, title=column)
    axis.tick_params(axis='x', rotation=60)
axes.ravel()[-1].axis('off'); plt.tight_layout(); plt.show()

## Возраст и пропуски пользователей

Возраст объединяется только с идентификатором клиента; доли пропусков считаются по исходному customer-справочнику.

In [ ]:
display(customers['age'].describe())
customers['age'].hist(bins=35, figsize=(10, 4)); plt.title('Возраст покупателей'); plt.show()
customer_missing = customers.isna().mean().mul(100).sort_values(ascending=False)
display(customer_missing.to_frame('missing_percent'))
transactions_with_age = transactions[['customer_id']].merge(
    customers[['customer_id', 'age']], on='customer_id', how='left'
)
print('Доля покупок без известного возраста, %:', round(transactions_with_age['age'].isna().mean() * 100, 2))

## Вывод

EDA количественно показывает long-tail товаров, разную активность клиентов, сезонность и пропуски профиля. Эти наблюдения объясняют необходимость popularity fallback, sparse-матриц и устойчивых к пропускам признаков.